## Imports and Configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import pickle
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Paths ---
# Works both as a .py script (__file__ defined) and in Jupyter (__file__ not defined).
# In Jupyter, place the notebook in the NCF/ folder so os.getcwd() points there.
try:
    _BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _BASE = os.getcwd()  # Jupyter: falls back to notebook's working directory

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/phase 3/feature_engineered_data"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/phase 3/NCF_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# --- Quick test mode ---
# Set True to verify pipeline with 5000 rows and 1 epoch (fast, no GPU needed)
QUICK_TEST = False

# --- Hyperparameters ---
EMBEDDING_DIM = 32  # Latent dim for GMF and MLP user/item embeddings (each)
MLP_LAYERS = [64, 32, 16]  # MLP hidden layer sizes after concat(user_emb, item_emb)
DROPOUT = 0.2
BATCH_SIZE = 1024
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 50 if not QUICK_TEST else 1
PATIENCE = 5  # Early stopping patience on HR@K
K = 10  # HR@K and NDCG@K
N_VAL_NEGATIVES = 99  # Negatives per positive in validation ranking set
VAL_SPLIT = 0.1  # Fraction of train positives held out for validation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"QUICK_TEST = {QUICK_TEST}")
print(
    f"Config: emb_dim={EMBEDDING_DIM}, mlp_layers={MLP_LAYERS}, "
    f"lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}"
)

DATA_DIR   : /content/drive/MyDrive/Colab Notebooks/phase 3/feature_engineered_data
OUTPUT_DIR : /content/drive/MyDrive/Colab Notebooks/phase 3/NCF_outputs
Using device: cpu
QUICK_TEST = False
Config: emb_dim=32, mlp_layers=[64, 32, 16], lr=0.001, batch=1024, epochs=50


## Load Data and Encode User/Item IDs

In [3]:
print("\n--- Loading parquet files ---")
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_with_features.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_with_features.parquet"))
neg_df = pd.read_parquet(os.path.join(DATA_DIR, "negatives_with_features.parquet"))

print(f"  train_with_features : {train_df.shape}")
print(f"  test_with_features  : {test_df.shape}")
print(f"  negatives           : {neg_df.shape}")

# Ensure the synthetic flag exists in all DataFrames
for df, name in [(train_df, "train"), (test_df, "test")]:
    if "is_synthetic_negative" not in df.columns:
        df["is_synthetic_negative"] = False
        print(f"  [info] Added is_synthetic_negative=False to {name}")

if "is_synthetic_negative" not in neg_df.columns:
    neg_df["is_synthetic_negative"] = True

# In QUICK_TEST mode, subsample to speed up verification
if QUICK_TEST:
    print("\n[QUICK_TEST] Subsampling data to 5000 rows each...")
    train_df = train_df.sample(
        n=min(5000, len(train_df)), random_state=SEED
    ).reset_index(drop=True)
    test_df = test_df.sample(n=min(2000, len(test_df)), random_state=SEED).reset_index(
        drop=True
    )
    neg_df = neg_df.sample(n=min(5000, len(neg_df)), random_state=SEED).reset_index(
        drop=True
    )

# Fit LabelEncoders on all known user/item IDs (train + test + negatives)
print("\n--- Encoding user/item IDs ---")
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

all_user_ids = pd.concat(
    [
        train_df["author_steamid"],
        test_df["author_steamid"],
        neg_df["author_steamid"],
    ]
).unique()

all_item_ids = pd.concat(
    [
        train_df["appid"],
        test_df["appid"],
        neg_df["appid"],
    ]
).unique()

user_encoder.fit(all_user_ids)
item_encoder.fit(all_item_ids)

N_USERS = len(user_encoder.classes_)
N_ITEMS = len(item_encoder.classes_)
print(f"  Unique users : {N_USERS:,}")
print(f"  Unique items : {N_ITEMS:,}")

# Apply encodings (add user_idx / item_idx columns)
for df in [train_df, test_df, neg_df]:
    df["user_idx"] = user_encoder.transform(df["author_steamid"])
    df["item_idx"] = item_encoder.transform(df["appid"])

# Save encoders for downstream phases (Phase 3 needs them to map embeddings back)
with open(os.path.join(OUTPUT_DIR, "user_encoder.pkl"), "wb") as f:
    pickle.dump(user_encoder, f)
with open(os.path.join(OUTPUT_DIR, "item_encoder.pkl"), "wb") as f:
    pickle.dump(item_encoder, f)
print("  Encoders saved to outputs/")


--- Loading parquet files ---
  train_with_features : (169439, 36)
  test_with_features  : (46922, 36)
  negatives           : (673048, 14)

--- Encoding user/item IDs ---
  Unique users : 32,792
  Unique items : 31,697
  Encoders saved to outputs/


## Build Training and Validation Sets

In [4]:
print("\n--- Building training and validation sets ---")

# Positive samples: real reviews where user recommended the game
train_pos = train_df[
    (train_df["voted_up"] == True) & (train_df["is_synthetic_negative"] == False)
][["user_idx", "item_idx"]].copy()
train_pos["label"] = 1.0
print(f"  Training positives : {len(train_pos):,}")

# Negative samples: synthetic genre-aware negatives (primary) + real downvotes
neg_synth = neg_df[["user_idx", "item_idx"]].copy()
neg_synth["label"] = 0.0

neg_real = train_df[
    (train_df["voted_up"] == False) & (train_df["is_synthetic_negative"] == False)
][["user_idx", "item_idx"]].copy()
neg_real["label"] = 0.0

all_negatives = (
    pd.concat([neg_synth, neg_real], ignore_index=True)
    .drop_duplicates(["user_idx", "item_idx"])
    .reset_index(drop=True)
)
print(f"  Total negatives    : {len(all_negatives):,}")

# Build user → negatives lookup for fast ranking-set construction
neg_by_user: dict[int, np.ndarray] = (
    all_negatives.groupby("user_idx")["item_idx"].apply(np.array).to_dict()
)

# Validation split: hold out VAL_SPLIT of positives (1 per user max, for clean HR@K)
train_pos_sorted = train_pos.drop_duplicates("user_idx", keep="first")
train_pos_multi = train_pos[~train_pos.index.isin(train_pos_sorted.index)]

# Split unique-per-user positives into train/val
pos_train_unique, pos_val_unique = train_test_split(
    train_pos_sorted, test_size=VAL_SPLIT, random_state=SEED, shuffle=True
)
# Keep all duplicate-user positives in training
train_pos_train = pd.concat([pos_train_unique, train_pos_multi], ignore_index=True)
train_pos_val = pos_val_unique.reset_index(drop=True)

print(f"  Train positives (for NCF training) : {len(train_pos_train):,}")
print(f"  Val positives   (for HR@K eval)    : {len(train_pos_val):,}")

# Build validation ranking set: 1 positive + up to N_VAL_NEGATIVES negatives per user
rng = np.random.default_rng(SEED)
val_records = []

for _, row in train_pos_val.iterrows():
    uid = int(row["user_idx"])
    pos_item = int(row["item_idx"])
    user_negs = neg_by_user.get(uid, np.array([], dtype=np.int64))

    # Exclude the positive item from negatives (safety check)
    user_negs = user_negs[user_negs != pos_item]
    if len(user_negs) == 0:
        continue

    n_sample = min(N_VAL_NEGATIVES, len(user_negs))
    sampled_negs = rng.choice(user_negs, size=n_sample, replace=False)

    val_records.append({"user_idx": uid, "item_idx": pos_item, "label": 1.0})
    for neg_item in sampled_negs:
        val_records.append({"user_idx": uid, "item_idx": int(neg_item), "label": 0.0})

val_df = pd.DataFrame(val_records)
print(f"  Validation records (pos + neg)     : {len(val_df):,}")

# Final training DataFrame: all positives + all negatives, shuffled
train_final = (
    pd.concat([train_pos_train, all_negatives], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(
    f"\n  Final training set : {len(train_final):,} rows  "
    f"(positive ratio: {train_final['label'].mean():.3f})"
)


--- Building training and validation sets ---
  Training positives : 126,673
  Total negatives    : 715,814
  Train positives (for NCF training) : 123,631
  Val positives   (for HR@K eval)    : 3,042
  Validation records (pos + neg)     : 63,793

  Final training set : 839,445 rows  (positive ratio: 0.147)


## NCF Dataset Class and DataLoaders

In [5]:
class NCFDataset(Dataset):
    """Simple Dataset: returns (user_idx, item_idx, label) integer/float tensors."""

    def __init__(self, df: pd.DataFrame):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.labels = torch.tensor(df["label"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]


train_dataset = NCFDataset(train_final)
val_dataset = NCFDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 4,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

Train batches : 820
Val   batches : 16


## NeuMF Model Architecture (GMF + MLP)

In [6]:
class NeuMF(nn.Module):
    """
    Neural Matrix Factorization — He et al. (2017).

    Two parallel pathways share no weights:
      GMF path  : element-wise product of dedicated user/item embeddings
                  → captures linear (bilinear) interactions
      MLP path  : concatenation of dedicated user/item embeddings through
                  a stack of fully-connected layers
                  → captures non-linear interactions

    Final output : sigmoid( W · [gmf_out ‖ mlp_out] + b )

    Saved embeddings (downstream):
      • GMF user/item embeddings (EMBEDDING_DIM each) → passed to Phase 3
      • These are the 32-dim collaborative filtering representations
    """

    def __init__(
        self,
        n_users: int,
        n_items: int,
        embedding_dim: int,
        mlp_layers: list[int],
        dropout: float,
    ):
        super().__init__()

        # GMF embeddings (separate from MLP — standard NeuMF design)
        self.gmf_user_emb = nn.Embedding(n_users, embedding_dim)
        self.gmf_item_emb = nn.Embedding(n_items, embedding_dim)

        # MLP embeddings
        self.mlp_user_emb = nn.Embedding(n_users, embedding_dim)
        self.mlp_item_emb = nn.Embedding(n_items, embedding_dim)

        # MLP tower: input = concat(user_emb, item_emb) = 2 * embedding_dim
        mlp_blocks = []
        in_dim = embedding_dim * 2
        for out_dim in mlp_layers:
            mlp_blocks += [
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = out_dim
        self.mlp = nn.Sequential(*mlp_blocks)

        # Output layer: GMF output (embedding_dim) ‖ MLP last layer (mlp_layers[-1]) → 1
        self.output_layer = nn.Linear(embedding_dim + mlp_layers[-1], 1)

        self._init_weights()

    def _init_weights(self):
        for emb in [
            self.gmf_user_emb,
            self.gmf_item_emb,
            self.mlp_user_emb,
            self.mlp_item_emb,
        ]:
            nn.init.normal_(emb.weight, std=0.01)
        for module in self.mlp:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        # GMF path
        gmf_u = self.gmf_user_emb(user_ids)  # (B, emb_dim)
        gmf_i = self.gmf_item_emb(item_ids)  # (B, emb_dim)
        gmf_out = gmf_u * gmf_i  # element-wise product

        # MLP path
        mlp_u = self.mlp_user_emb(user_ids)  # (B, emb_dim)
        mlp_i = self.mlp_item_emb(item_ids)  # (B, emb_dim)
        mlp_out = self.mlp(torch.cat([mlp_u, mlp_i], dim=-1))  # (B, mlp_layers[-1])

        # Combine and predict
        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        logits = self.output_layer(combined).squeeze(-1)
        return torch.sigmoid(logits)


model = NeuMF(
    n_users=N_USERS,
    n_items=N_ITEMS,
    embedding_dim=EMBEDDING_DIM,
    mlp_layers=MLP_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

NeuMF(
  (gmf_user_emb): Embedding(32792, 32)
  (gmf_item_emb): Embedding(31697, 32)
  (mlp_user_emb): Embedding(32792, 32)
  (mlp_item_emb): Embedding(31697, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)

Trainable parameters: 4,134,113


## Evaluation Functions (HR@K and NDCG@K)

In [7]:
def predict_scores(
    model: NeuMF,
    user_idx: np.ndarray,
    item_idx: np.ndarray,
    device: torch.device,
    chunk_size: int = 8192,
) -> np.ndarray:
    """Run batched inference to avoid OOM on large ranking sets."""
    model.eval()
    u_t = torch.tensor(user_idx, dtype=torch.long)
    i_t = torch.tensor(item_idx, dtype=torch.long)
    scores = []
    with torch.no_grad():
        for start in range(0, len(u_t), chunk_size):
            u_chunk = u_t[start : start + chunk_size].to(device)
            i_chunk = i_t[start : start + chunk_size].to(device)
            scores.append(model(u_chunk, i_chunk).cpu().numpy())
    return np.concatenate(scores)


def compute_hr_ndcg(
    model: NeuMF,
    eval_df: pd.DataFrame,
    device: torch.device,
    k: int = 10,
) -> tuple[float, float]:
    """
    Compute HR@K and NDCG@K.

    eval_df must contain one row with label=1 per user (the target positive)
    and N rows with label=0 per user (sampled negatives).
    """
    scores = predict_scores(
        model,
        eval_df["user_idx"].values,
        eval_df["item_idx"].values,
        device,
    )
    df = eval_df[["user_idx", "item_idx", "label"]].copy()
    df["score"] = scores

    hits, ndcgs = [], []
    for uid, group in df.groupby("user_idx"):
        pos_rows = group[group["label"] == 1.0]
        if len(pos_rows) == 0:
            continue
        pos_item = pos_rows["item_idx"].values[0]
        ranked = group.sort_values("score", ascending=False)["item_idx"].values
        top_k = ranked[:k]

        if pos_item in top_k:
            rank = np.where(ranked == pos_item)[0][0] + 1  # 1-indexed
            hits.append(1)
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return float(np.mean(hits)), float(np.mean(ndcgs))


# Sanity-check: one forward pass before training
_u = torch.zeros(4, dtype=torch.long).to(DEVICE)
_i = torch.zeros(4, dtype=torch.long).to(DEVICE)
with torch.no_grad():
    _out = model(_u, _i)
print(f"Sanity-check output shape: {_out.shape}, values: {_out.tolist()}")
assert _out.shape == (4,), "Unexpected output shape"
print("Forward pass OK.")

Sanity-check output shape: torch.Size([4]), values: [0.5012850761413574, 0.5003066062927246, 0.4996129870414734, 0.5008554458618164]
Forward pass OK.


## Training Loop with Early Stopping (on validation HR@K)

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_hr = 0.0
best_ndcg = 0.0
best_epoch = 0
patience_counter = 0
train_losses = []
val_hrs = []
val_ndcgs = []

CKPT_PATH = os.path.join(OUTPUT_DIR, "ncf_best_model.pt")

print(f"\n--- Training NeuMF (up to {EPOCHS} epochs, patience={PATIENCE}) ---")
print(f"{'Epoch':>6}  {'Loss':>8}  {'HR@10':>8}  {'NDCG@10':>8}  {'':10}")

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    total_loss = 0.0
    for u, i, y in train_loader:
        u, i, y = u.to(DEVICE), i.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(u, i), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ---- Validate ----
    hr, ndcg = compute_hr_ndcg(model, val_df, DEVICE, k=K)
    val_hrs.append(hr)
    val_ndcgs.append(ndcg)

    note = ""
    if hr > best_hr:
        best_hr, best_ndcg, best_epoch = hr, ndcg, epoch
        torch.save(model.state_dict(), CKPT_PATH)
        patience_counter = 0
        note = "<-- best"
    else:
        patience_counter += 1
        note = f"patience {patience_counter}/{PATIENCE}"

    print(f"{epoch:>6}  {avg_loss:>8.4f}  {hr:>8.4f}  {ndcg:>8.4f}  {note}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

print(
    f"\nBest  HR@{K}: {best_hr:.4f}   NDCG@{K}: {best_ndcg:.4f}   (epoch {best_epoch})"
)


--- Training NeuMF (up to 50 epochs, patience=5) ---
 Epoch      Loss     HR@10   NDCG@10            
     1    0.4317    0.7498    0.3891  <-- best
     2    0.4009    0.7788    0.4129  <-- best
     3    0.3734    0.8064    0.4546  <-- best
     4    0.3388    0.8373    0.4939  <-- best
     5    0.3044    0.8537    0.5227  <-- best
     6    0.2745    0.8580    0.5440  <-- best
     7    0.2485    0.8652    0.5611  <-- best
     8    0.2238    0.8715    0.5750  <-- best
     9    0.2005    0.8774    0.5886  <-- best
    10    0.1784    0.8777    0.5984  <-- best
    11    0.1590    0.8803    0.6105  <-- best
    12    0.1408    0.8823    0.6156  <-- best
    13    0.1260    0.8761    0.6257  patience 1/5
    14    0.1134    0.8757    0.6235  patience 2/5
    15    0.1024    0.8754    0.6266  patience 3/5
    16    0.0936    0.8803    0.6322  patience 4/5
    17    0.0857    0.8771    0.6335  patience 5/5

Early stopping triggered at epoch 17.

Best  HR@10: 0.8823   NDCG@10: 0.6156 

## Test Evaluation and Save Embeddings / Scores

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print(f"Loaded best checkpoint from epoch {best_epoch}.")

# ---- Build test ranking set (same structure as validation) ----
test_pos = test_df[
    (test_df["voted_up"] == True) & (test_df["is_synthetic_negative"] == False)
][["user_idx", "item_idx"]].copy()
test_pos["label"] = 1.0

# One positive per user for clean HR@K
test_pos_unique = test_pos.drop_duplicates("user_idx", keep="first")

test_records = []
for _, row in test_pos_unique.iterrows():
    uid = int(row["user_idx"])
    pos_item = int(row["item_idx"])
    user_negs = neg_by_user.get(uid, np.array([], dtype=np.int64))
    user_negs = user_negs[user_negs != pos_item]
    if len(user_negs) == 0:
        continue
    n_sample = min(N_VAL_NEGATIVES, len(user_negs))
    sampled_negs = rng.choice(user_negs, size=n_sample, replace=False)
    test_records.append({"user_idx": uid, "item_idx": pos_item, "label": 1.0})
    for neg_item in sampled_negs:
        test_records.append({"user_idx": uid, "item_idx": int(neg_item), "label": 0.0})

test_eval_df = pd.DataFrame(test_records)
print(f"Test ranking records: {len(test_eval_df):,}")

test_hr, test_ndcg = compute_hr_ndcg(model, test_eval_df, DEVICE, k=K)
print(f"\nTest  HR@{K}  : {test_hr:.4f}")
print(f"Test  NDCG@{K}: {test_ndcg:.4f}")

# ---- Extract MLP embeddings (32-dim) → features for Phase 3 ----
# NOTE: GMF embeddings collapse to ~0 during joint training (known NeuMF issue).
# MLP embeddings train normally and carry the collaborative signal.
mlp_user_emb_np = model.mlp_user_emb.weight.detach().cpu().numpy()  # (N_USERS, 32)
mlp_item_emb_np = model.mlp_item_emb.weight.detach().cpu().numpy()  # (N_ITEMS, 32)

emb_cols_user = [f"ncf_user_emb_{j}" for j in range(EMBEDDING_DIM)]
emb_cols_item = [f"ncf_item_emb_{j}" for j in range(EMBEDDING_DIM)]

user_emb_df = pd.DataFrame(mlp_user_emb_np, columns=emb_cols_user)
user_emb_df.insert(0, "user_idx", np.arange(N_USERS))
user_emb_df.insert(
    1, "author_steamid", user_encoder.inverse_transform(np.arange(N_USERS))
)

item_emb_df = pd.DataFrame(mlp_item_emb_np, columns=emb_cols_item)
item_emb_df.insert(0, "item_idx", np.arange(N_ITEMS))
item_emb_df.insert(1, "appid", item_encoder.inverse_transform(np.arange(N_ITEMS)))

user_emb_path = os.path.join(OUTPUT_DIR, "ncf_user_embeddings.parquet")
item_emb_path = os.path.join(OUTPUT_DIR, "ncf_item_embeddings.parquet")
user_emb_df.to_parquet(user_emb_path, index=False)
item_emb_df.to_parquet(item_emb_path, index=False)
print(f"\nUser embeddings saved : {user_emb_df.shape}  ->{user_emb_path}")
print(f"Item embeddings saved : {item_emb_df.shape}  ->{item_emb_path}")

# ---- Compute and save NCF scores for all train/test rows (Phase 4 input) ----
print("\nComputing NCF scores for train + test rows (Phase 4 input)...")


def score_dataframe(
    df: pd.DataFrame, model: NeuMF, device: torch.device
) -> pd.DataFrame:
    """Compute NCF scores for every row in df and return a lightweight DataFrame."""
    scores = predict_scores(
        model,
        df["user_idx"].values,
        df["item_idx"].values,
        device,
    )
    return pd.DataFrame(
        {
            "author_steamid": df["author_steamid"].values,
            "appid": df["appid"].values,
            "user_idx": df["user_idx"].values,
            "item_idx": df["item_idx"].values,
            "voted_up": df["voted_up"].values,
            "ncf_score": scores,
        }
    )


train_scores = score_dataframe(train_df, model, DEVICE)
test_scores = score_dataframe(test_df, model, DEVICE)
neg_scores = score_dataframe(neg_df, model, DEVICE)

train_scores_path = os.path.join(OUTPUT_DIR, "ncf_train_scores.parquet")
test_scores_path = os.path.join(OUTPUT_DIR, "ncf_test_scores.parquet")
neg_scores_path = os.path.join(OUTPUT_DIR, "ncf_neg_scores.parquet")

train_scores.to_parquet(train_scores_path, index=False)
test_scores.to_parquet(test_scores_path, index=False)
neg_scores.to_parquet(neg_scores_path, index=False)

print(f"NCF train scores : {train_scores.shape}  ->{train_scores_path}")
print(f"NCF test scores  : {test_scores.shape}   ->{test_scores_path}")
print(f"NCF neg scores   : {neg_scores.shape}    ->{neg_scores_path}")

# ---- Summary ----
print("\n" + "=" * 60)
print("Phase 1 Complete - outputs/")
print("=" * 60)
print(f"  ncf_best_model.pt            Model checkpoint (epoch {best_epoch})")
print(
    f"  ncf_user_embeddings.parquet  {N_USERS} users x{EMBEDDING_DIM} dims  (Phase 3)"
)
print(
    f"  ncf_item_embeddings.parquet  {N_ITEMS} items x{EMBEDDING_DIM} dims  (Phase 3)"
)
print(f"  ncf_train_scores.parquet     NCF scores for train rows  (Phase 4)")
print(f"  ncf_test_scores.parquet      NCF scores for test rows   (Phase 4)")
print(f"  ncf_neg_scores.parquet       NCF scores for negatives   (Phase 4)")
print(f"  user_encoder.pkl             LabelEncoder: steamid -> user_idx")
print(f"  item_encoder.pkl             LabelEncoder: appid   -> item_idx")
print(f"\nBest val  HR@{K}: {best_hr:.4f}  NDCG@{K}: {best_ndcg:.4f}")
print(f"Test      HR@{K}: {test_hr:.4f}  NDCG@{K}: {test_ndcg:.4f}")